In [1]:
pip install pandas numpy scikit-learn xgboost joblib

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   - -------------------------------------- 2.6/69.5 MB 14.3 MB/s eta 0:00:05
   --- ------------------------------------ 5.8/69.5 MB 14.8 MB/s eta 0:00:05
   ----- ---------------------------------- 9.7/69.5 MB 16.2 MB/s eta 0:00:04
   ------- -------------------------------- 13.1/69.5 MB 16.6 MB/s eta 0:00:04
   --------- ------------------------------ 16.8/69.5 MB 17.1 MB/s eta 0:00:04
   ------------ --------------------------- 21.5/69.5 MB 17.6 MB/s eta 0:00:03
   --------------- ------------------------ 27.0/69.5 MB 18.9 MB/s eta 0:00:03
   ------------------ --------------------- 31.5/69.5 MB 19.3 MB/s eta 0:00:02
   -------------------- ------------------- 35.4/69.5 MB 19.3 MB/s eta 0:00:02
   ---------------------- ----------------- 39.6/69.5 MB 19.3 MB/s eta 0:00:02
   ------------------------ --------------- 43.3/69.5 MB 19.3 MB/s eta 0:00:02
   --------------------------- ------------ 47.4/69.5 MB 19.3 MB


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: C:\Users\chockalingam\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Load Dataset
data_path = 'Crop_recommendation.csv'
df = pd.read_csv(data_path)

# 2. Separate Features and Target
X = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y = df['label']

# 3. Encode Categorical Labels for XGBoost (XGBoost requires numeric targets 0 to N-1)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 4. Train-Test Split (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# 5. Fit Feature Scaler on Training Data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Define Base Estimators for the Ensemble
base_estimators = [
    ('random_forest', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('xgboost', XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, eval_metric='mlogloss', n_jobs=-1))
]

# 7. Define Meta-Learner and Combine Into Stacking Classifier
# This acts as our 'one more model combination' layer to blend predictions smoothly
meta_learner = LogisticRegression(max_iter=500, random_state=42)

ensemble_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=5,
    n_jobs=-1
)

# 8. Train the Hybrid Model
print("Training the production ensemble model...")
ensemble_model.fit(X_train_scaled, y_train)

# 9. Evaluate Performance
y_pred = ensemble_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"\n Validation Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report Summary:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# 10. CRITICAL STEP: Save artifact bundle to prevent deployment scale shifts
joblib.dump(ensemble_model, 'crop_ensemble_model.pkl')
joblib.dump(scaler, 'crop_scaler.pkl')
joblib.dump(label_encoder, 'crop_label_encoder.pkl')
print("\n All assets ('crop_ensemble_model.pkl', 'crop_scaler.pkl', 'crop_label_encoder.pkl') saved successfully!")

Training the production ensemble model...

 Validation Accuracy: 99.55%

Classification Report Summary:
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      1.00      1.00        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00      0.98        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      0.95      0.97        20
       maize       1.00      1.00      1.00        20
       mango       1.00      1.00      1.00        20
   mothbeans       0.95      1.00      0.98        20
    mungbean       1.00      1.00      1.00        20
   muskmelon       1.00      1.

In [3]:
import joblib
import numpy as np

# 1. Load the pre-trained asset artifacts
try:
    model = joblib.load('crop_ensemble_model.pkl')
    scaler = joblib.load('crop_scaler.pkl')
    label_encoder = joblib.load('crop_label_encoder.pkl')
    print("All ML assets successfully loaded into runtime context.")
except FileNotFoundError as e:
    print(f"Asset loading exception: {e}. Please execute train.py first.")
    exit()

def predict_crop(n, p, k, temp, hum, ph, rain):
    """
    Accepts raw environmental features, structures them into a vector matrix,
    applies calibration scaling transformations, and returns a verified crop string.
    """
    # Defensive programming block: Check for outlier sensor anomalies
    if n > 140 or k > 205:
        return {
            "status": "error",
            "message": f"Input value out of agricultural range (Detected N={n}, K={k}). Please check sensor calibration parameters."
        }
    
    # Structure inputs into a uniform 2D numpy matrix format matching training structures
    raw_features = np.array([[n, p, k, temp, hum, ph, rain]])
    
    # Step A: Standardize incoming raw matrices using our saved standard scaler parameters
    scaled_features = scaler.transform(raw_features)
    
    # Step B: Pass standardized array through the Stacking Classifier pipeline
    numeric_prediction = model.predict(scaled_features)
    
    # Step C: Decode numeric categorical indices back to original string identities
    predicted_crop = label_encoder.inverse_transform(numeric_prediction)[0]
    
    return {
        "status": "success",
        "recommended_crop": predicted_crop
    }

# --- Runtime Execution Test Box ---
if __name__ == "__main__":
    # Test Scenario 1: Providing a realistic input profile for Papaya/Rice tropical ranges
    sample_input = {
        "N": 50, "P": 60, "K": 50,
        "temperature": 27.2, "humidity": 92.0, "ph": 6.7, "rainfall": 214.8
    }
    
    result = predict_crop(
        sample_input["N"], sample_input["P"], sample_input["K"],
        sample_input["temperature"], sample_input["humidity"],
        sample_input["ph"], sample_input["rainfall"]
    )
    print("\n[Test Result]:", result)

All ML assets successfully loaded into runtime context.

[Test Result]: {'status': 'success', 'recommended_crop': 'papaya'}


C:\Users\chockalingam\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [12]:
import joblib
import numpy as np

# 1. Load the ML files (Your model pipeline assets)
try:
    scaler = joblib.load('crop_scaler.pkl')
    model = joblib.load('crop_ensemble_model.pkl')
    label_encoder = joblib.load('crop_label_encoder.pkl')
    print("✅ All ML assets loaded successfully!\n")
except FileNotFoundError as e:
    print(f"❌ Error loading files: {e}. Ensure the .pkl files are in this directory.")
    exit()

def run_prediction_pipeline(test_case_name, raw_data):
    """
    Executes the step-by-step pipeline using the three loaded pkl assets.
    """
    print(f"--- Running Pipeline for: {test_case_name} ---")
    print(f"Raw Input Sensor Data: {raw_data}")
    
    # Validation step: Protect against out-of-bounds sensor anomalies
    if raw_data["N"] > 140 or raw_data["K"] > 205:
        print("❌ Result: Blocked! Value is outside realistic training data parameters.\n")
        return

    # Step 1: Format into a 2D Array matching the training layout
    features_array = np.array([[
        raw_data["N"], raw_data["P"], raw_data["K"],
        raw_data["temperature"], raw_data["humidity"], 
        raw_data["ph"], raw_data["rainfall"]
    ]])
    
    # Step 2: Use crop_scaler.pkl to scale the features
    scaled_features = scaler.transform(features_array)
    
    # Step 3: Use crop_ensemble_model.pkl to get the numeric class prediction
    numeric_prediction = model.predict(scaled_features)
    
    # Step 4: Use crop_label_encoder.pkl to translate the number to a crop name
    final_crop = label_encoder.inverse_transform(numeric_prediction)[0]
    
    print(f"🎯 Recommended Crop Result: {final_crop.upper()}\n")


# =====================================================================
# DEFINING THE TEST CASES
# =====================================================================

# Test Case 1: Perfectly calibrated profile for Papaya
# (High temperature, extreme humidity, moderate-heavy rainfall)
papaya_test = {
    "N": 105, "P": 82, "K": 52, 
    "temperature": 32.8, "humidity": 68.2, "ph": 6.5, "rainfall": 95.5
}

# Test Case 2: Calibrated profile for Maize (Corn)
# (High Nitrogen, lower humidity, moderate rainfall)
maize_test = {
    "N": 75, "P": 40, "K": 22,
    "temperature": 23.8, "humidity": 65.4, "ph": 6.2, "rainfall": 90.1
}

# Test Case 3: Anomaly / Out-of-Bounds Error Data
# (Simulating the exact data value mismatch error from your original JSON)
mismatch_error_test = {
    "N": 100, "P": 28, "K": 30, 
    "temperature": 24.2, "humidity": 56.5, "ph": 6.2, "rainfall": 145.0
}

# =====================================================================
# EXECUTION
# =====================================================================
if __name__ == "__main__":
    run_prediction_pipeline("Test Case 1 (Papaya Profile)", papaya_test)
    #run_prediction_pipeline("Test Case 2 (Maize Profile)", maize_test)
    #run_prediction_pipeline("Test Case 3 (Anomaly Error Profile)", mismatch_error_test)

✅ All ML assets loaded successfully!

--- Running Pipeline for: Test Case 1 (Papaya Profile) ---
Raw Input Sensor Data: {'N': 105, 'P': 82, 'K': 52, 'temperature': 32.8, 'humidity': 68.2, 'ph': 6.5, 'rainfall': 95.5}


C:\Users\chockalingam\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


🎯 Recommended Crop Result: BANANA

